# 集群平台、网络存储与可靠性补充线 · 第 5/8 课：Gang Scheduling 与拓扑感知放置

> 状态：**未开始**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现单 rack 内的 all-or-nothing gang 放置，并解释 pending、碎片与训练通信性能。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/` 假定 GPU 已获得；本课解释调度器如何同时拿到整组资源，并把通信密集 ranks 放在合适拓扑域。

前置：Linux/网络基础、runtime 补充线、train 分布式章节。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Gang scheduling 要求至少 minCount Pods 作为一组原子满足；topology-aware placement 在 rack/zone/node label 上寻找可容纳整组的域，再绑定。

### 数据与控制如何流动

调度周期先对整组做容量与约束过滤，再按 rack/node/NIC 拓扑评分并保留资源；只有 minCount 全部可绑定时才提交，否则整体等待并释放临时假设，避免半组占坑。

### 正确性条件与常见误区

只调度一半 workers 会占资源却无法开始 collective；拓扑标签必须可信且容量要考虑 GPU 型号、健康、NIC/rail 和 NUMA，不是只数 GPU。

### 性能、成本与工程取舍

严格单 rack 性能好但排队长；允许跨 rack 提高可调度性却可能降低通信吞吐。过大的 gang 容易造成资源碎片和 starvation。

## 具体演示

需要 8 GPU，rack A 节点空闲 [4,4] 可放置；rack B [6,1,1] 也总数 8，但若要求单节点最小 2 或同 rail，还需额外约束。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐最小 rack 名称的 gang 放置；不能跨 rack 拼接。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def place_gang(free_gpus_by_rack, required_gpus):
    if required_gpus <= 0:
        raise ValueError("gang size must be positive")
    candidates = []
    for rack, node_free in free_gpus_by_rack.items():
        if sum(node_free) >= required_gpus:
            candidates.append(rack)
    # TODO：无候选返回 None，否则确定性选择字典序最小 rack。
    return ______

assert place_gang({"rack-b": [8], "rack-a": [4, 4]}, 8) == "rack-a"
assert place_gang({"rack-a": [4], "rack-b": [3, 1]}, 8) is None


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么普通逐 Pod 调度会让分布式训练死锁式占坑？

**你的答案：**


### Q2

拓扑约束越严格越好吗？

**你的答案：**


### Q3

节点有 8 GPU 但其中一张 ECC unhealthy，gang size=8 应如何处理？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考资料

- [Kubernetes Gang Scheduling](https://kubernetes.io/docs/concepts/scheduling-eviction/gang-scheduling/)
- [Kubernetes Topology Manager](https://kubernetes.io/docs/tasks/administer-cluster/topology-manager/)
- [Kubernetes Scheduling Framework](https://kubernetes.io/docs/concepts/scheduling-eviction/scheduling-framework/)

API 与平台能力会演进；部署前应按目标版本重新核对。